# TP2 — Classification : quels clients vont résilier ? (Énoncé)

**Durée : 2 h** · Cours associé : [TP2_Cours.md](TP2_Cours.md) · Pense-bête : [Pense_bete_Pandas.md](../Pense_bete_Pandas.md)

## Contexte

Un opérateur de télécommunications perd chaque mois une partie de ses abonnés : c'est l'**attrition** (en anglais *churn*). Retenir un client coûte beaucoup moins cher que d'en conquérir un nouveau : le service marketing veut donc repérer **à l'avance** les clients qui risquent de partir, pour leur proposer une offre adaptée.

Le fichier `data/telco_churn.csv` décrit environ 7 000 clients : leur contrat, leur ancienneté, leur facture, les services souscrits… et indique s'ils ont résilié leur abonnement (`Churn = Yes`).

**Objectif** : construire un modèle qui prédit si un client va résilier, et évaluer honnêtement ses performances.

## Déroulé

| Partie | Contenu | Durée indicative |
|---|---|---|
| 1 | Exploration des données | 20 min |
| 2 | Découpage train / test et baseline | 10 min |
| 3 | Pipeline de préparation et régression logistique | 25 min |
| 4 | Métriques d'évaluation | 20 min |
| 5 | k-NN, arbre de décision et sur-apprentissage | 25 min |
| 6 | Validation croisée et choix du modèle | 20 min |

Les cellules marquées `# TODO` sont à compléter. Les cellules ✍️ attendent une réponse rédigée.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)

plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42

## Partie 1 — Exploration des données (20 min)

| Colonne | Signification |
|---|---|
| `customerID` | identifiant du client |
| `gender`, `SeniorCitizen`, `Partner`, `Dependents` | sexe, plus de 65 ans (1/0), en couple, personnes à charge |
| `tenure` | ancienneté, en mois |
| `PhoneService`, `MultipleLines` | téléphone, plusieurs lignes |
| `InternetService` | type d'accès : `DSL`, `Fiber optic` ou `No` |
| `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport` | options de sécurité, sauvegarde, protection, assistance |
| `StreamingTV`, `StreamingMovies` | options de streaming |
| `Contract` | type de contrat : au mois, un an ou deux ans |
| `PaperlessBilling`, `PaymentMethod` | facture dématérialisée, mode de paiement |
| `MonthlyCharges`, `TotalCharges` | facture mensuelle et total facturé depuis l'abonnement ($) |
| `Churn` | le client a-t-il résilié ? `Yes` ou `No` (**cible**) |

**1.1** Chargez les données dans `df`, affichez les dimensions, les premières lignes et les types des colonnes (`dtypes`). Une colonne numérique a-t-elle un type surprenant ?

In [ ]:
# TODO : chargement, dimensions, types


**1.2** La colonne `TotalCharges` est stockée comme du texte. Convertissez-la en nombre avec `pd.to_numeric(..., errors="coerce")` : les valeurs impossibles à convertir deviennent `NaN`. Combien y en a-t-il ? Regardez l'ancienneté (`tenure`) de ces clients : pourquoi n'ont-ils pas de montant total ?

In [ ]:
# TODO : conversion de TotalCharges et étude des valeurs manquantes


✍️ **Votre réponse :**

…

**1.3** Quel est le taux de résiliation global ? Calculez ensuite le taux de résiliation selon le type de contrat (`Contract`), le type d'accès internet (`InternetService`) et le mode de paiement (`PaymentMethod`).

*Astuce : créez d'abord une colonne numérique `churn` valant 1 si `Churn == "Yes"`, 0 sinon ; son taux moyen est alors le taux de résiliation.*

In [ ]:
# TODO : taux de résiliation global et par groupe


**1.4** Comparez l'ancienneté (`tenure`) des clients qui partent et de ceux qui restent (deux histogrammes superposés), puis leur facture mensuelle moyenne.

In [ ]:
# TODO : ancienneté et facture selon la résiliation


✍️ **Votre réponse :**

…

**1.5** Quelle colonne faut-il exclure des variables explicatives ? Pourquoi ?

✍️ **Votre réponse :**

…

## Partie 2 — Découpage train / test et baseline (10 min)

**2.1** Construisez `X` (toutes les colonnes sauf `customerID`, `Churn` et `churn`) et `y` (`churn`), puis découpez en 80 % entraînement / 20 % test, de façon **stratifiée**, avec `random_state=RANDOM_STATE`. Vérifiez que le taux de résiliation est le même dans les deux jeux.

In [ ]:
# TODO : X, y et découpage stratifié


**2.2** Entraînez un `DummyClassifier(strategy="most_frequent")` et calculez son exactitude sur le jeu de test. Que prédit-il ? Ce modèle est-il utile au service marketing ?

In [ ]:
# TODO : baseline


✍️ **Votre réponse :**

…

## Partie 3 — Pipeline de préparation et régression logistique (25 min)

**3.1** Construisez un `ColumnTransformer` nommé `preprocess` qui :

- pour les colonnes numériques `tenure`, `MonthlyCharges`, `TotalCharges`, `SeniorCitizen` : impute par la **médiane** puis **standardise** ;
- pour toutes les autres colonnes (catégorielles) : impute par la **modalité la plus fréquente** puis applique un **one-hot encoding** (`handle_unknown="ignore"`).

*Astuce : `X.select_dtypes(exclude="number").columns` donne la liste des colonnes non numériques.*

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
# TODO : liste des colonnes catégorielles et ColumnTransformer

**3.2** Pour comprendre ce que fait `preprocess`, appliquez-le au jeu d'entraînement (`fit_transform`) : combien de colonnes obtient-on ? Affichez le résultat sous forme de DataFrame avec les noms de colonnes produits (`get_feature_names_out()`).

In [ ]:
# TODO : observer la sortie du préprocesseur


**3.3** Créez un `Pipeline` `logreg` qui enchaîne `preprocess` et une `LogisticRegression(max_iter=1000)`. Entraînez-le et affichez l'exactitude sur le jeu d'entraînement et sur le jeu de test.

In [ ]:
# TODO : pipeline régression logistique


**3.4** Affichez les 8 coefficients les plus positifs et les 8 plus négatifs de la régression logistique, en face des noms de variables. Quels facteurs augmentent le risque de départ ? Lesquels le diminuent ? Est-ce cohérent avec la partie 1 ?

*Indice : `logreg.named_steps["clf"].coef_[0]` et `logreg.named_steps["prep"].get_feature_names_out()`.*

In [ ]:
# TODO : coefficients


✍️ **Votre réponse :**

…

## Partie 4 — Métriques d'évaluation (20 min)

**4.1** Affichez la matrice de confusion de `logreg` sur le jeu de test (`ConfusionMatrixDisplay.from_estimator`) et le `classification_report`. Parmi les clients qui sont vraiment partis, combien le modèle en a-t-il « ratés » ?

In [ ]:
# TODO : matrice de confusion et rapport


**4.2** Recalculez **à la main** (à partir des quatre cases de la matrice) la précision et le rappel de la classe « part », puis vérifiez avec `precision_score` et `recall_score`. Traduisez chacun en une phrase pour le service marketing.

In [ ]:
# TODO : précision et rappel à la main


**4.3** Tracez la courbe ROC et calculez l'AUC. Puis, en utilisant `predict_proba`, calculez précision et rappel pour des seuils de décision de 0,3, 0,5 et 0,7. Quel seuil choisiriez-vous si une offre de fidélité coûte peu et qu'un client perdu coûte cher ?

In [ ]:
# TODO : ROC, AUC et effet du seuil


✍️ **Votre réponse :**

…

## Partie 5 — k-NN, arbre de décision et sur-apprentissage (25 min)

**5.1** Créez un pipeline `knn` (même `preprocess` + `KNeighborsClassifier(n_neighbors=5)`) et un pipeline `arbre` (`preprocess` + `DecisionTreeClassifier(random_state=RANDOM_STATE)` sans limite de profondeur). Pour chacun, affichez l'exactitude en entraînement et en test.

*Astuce : réutiliser le même objet `preprocess` dans plusieurs pipelines pose problème (il serait partagé). Utilisez `sklearn.base.clone(preprocess)`.*

In [ ]:
from sklearn.base import clone
# TODO : k-NN et arbre non contraint

**5.2** Faites varier la profondeur maximale de l'arbre (`max_depth` de 1 à 15) et tracez l'exactitude en entraînement et en test en fonction de la profondeur. Identifiez les zones de sous-apprentissage et de sur-apprentissage.

*Astuce : `arbre.set_params(clf__max_depth=d)` modifie un hyperparamètre d'une étape du pipeline.*

In [ ]:
# TODO : courbe de complexité de l'arbre


✍️ **Votre réponse :**

…

**5.3** Entraînez un arbre de profondeur 3 et affichez-le avec `plot_tree` (paramètres utiles : `feature_names`, `class_names`, `filled=True`). Quelles sont les premières questions posées par l'arbre ?

In [ ]:
# TODO : visualiser un arbre peu profond


✍️ **Votre réponse :**

…

## Partie 6 — Validation croisée et choix du modèle (20 min)

**6.1** Évaluez les trois pipelines (`logreg`, `knn` et `arbre` avec `max_depth=None`) par **validation croisée à 5 plis sur le jeu d'entraînement** (`cross_val_score`), en utilisant cette fois l'**AUC** comme métrique (`scoring="roc_auc"`), mieux adaptée aux classes déséquilibrées. Affichez la moyenne et l'écart-type pour chacun.

In [ ]:
# TODO : validation croisée des trois modèles


**6.2** Utilisez `GridSearchCV` (`scoring="roc_auc"`) pour régler simultanément `max_depth` (2 à 10) et `min_samples_leaf` (1, 10, 25, 50) de l'arbre, en validation croisée à 5 plis. Affichez les meilleurs hyperparamètres et le score correspondant.

In [ ]:
# TODO : recherche d'hyperparamètres pour l'arbre


**6.3** Faites de même pour le k-NN avec `n_neighbors` de 5 à 100 (par pas de 5).

In [ ]:
# TODO : recherche d'hyperparamètres pour le k-NN


**6.4** Choisissez **un** modèle final à partir des scores de validation croisée, puis évaluez-le **une seule fois** sur le jeu de test (AUC, précision et rappel). Concluez pour le service marketing.

In [ ]:
# TODO : évaluation finale


✍️ **Votre réponse :**

…

## Pour aller plus loin (bonus)

**Choisir le seuil selon les coûts.** Supposons qu'une offre de fidélité coûte **50 $** par client contacté, et qu'un client perdu coûte **500 $** (le chiffre d'affaires qu'il aurait rapporté). On suppose aussi que l'offre suffit à retenir un client qui allait partir.

1. Pour des seuils de 0,05 à 0,95, calculez le coût total sur le jeu de test : 50 $ par client contacté (prédit « part ») + 500 $ par départ non repéré (faux négatif).
2. Tracez le coût en fonction du seuil. Quel seuil minimise le coût ? Combien économise-t-on par rapport au seuil de 0,5, et par rapport à « ne rien faire » ?

*Remarque : en toute rigueur, le seuil devrait être choisi sur des données de validation, pas sur le jeu de test.*

In [ ]:
# Bonus : seuil qui minimise le coût


✍️ **Votre réponse :**

…